# A Simulation Study {#sec-simulation-anova}



In [1]:
## Warning: to compile the notes you need the "bookdown" and the "broom" packages. Install them by
## running install.packages, see the commented lines below

if (!require("tidyverse")) {
  install.packages("tidyverse")
}

if (!require("broom")) {
  install.packages("broom")
}

if (!require("latex2exp")) {
  install.packages("latex2exp")
}

suppressMessages(library(tidyverse))
suppressMessages(library(broom))

kids <- read_csv(
  "https://raw.githubusercontent.com/feb-uni-sofia/econometrics2020-solutions/master/data/childiq.csv") %>%
  select(kid_score, mom_hs)

Loading required package: tidyverse

Warning message:
“package ‘tidyverse’ was built under R version 4.4.3”
Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Warning message:
“package ‘tibble’ was built under R version 4.4.3”
Warning message:
“package ‘tidyr’ was built under R version 4.4.3”
Warning message:
“package ‘readr’ was built under R version 4.4.3”
Warning message:
“package ‘purrr’ was built under R version 4.4.3”
Warning message:
“package ‘dplyr’ was built under R version 4.4.3”
Warning message:
“package ‘stringr’ was built under R version 4.4.3”
Warning message:
“package ‘forcats’ was built under R version 4.4.3”
Warning message:
“package ‘lubridate’ was built under R version 4.4.3”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ─



In the previous section we had a sample of children and tried to figure out how the (unobserved) population of children might look like. We used OLS to compute estimates for the population coefficients and tested a hypothesis about the difference of population group averages (expected values).

In the present section we will reverse the process. We will take a population (a model with known coefficients) and we will generate random samples from that population in order to see how and why the statistical machinery works. The problem that we want to research is basically: if we know the population, how does the data look like when we start selecting samples from this population.

```{mermaid}
flowchart LR
  A[Population] --> B(Sample 1)
  A --> C(Sample 2)
  A --> D(Sample R)
  B --> B1[OLS estimates in sample 1]
  C --> C1[OLS estimates in sample 2]
  D --> D1[OLS estimates in sample R]
```

## The population

In order to study the statistical properties of the OLS estimators $\hat{\beta}_0$ and $\hat{\beta}_1$ we will generate a large number of samples from the following model.

$$
\begin{align*}
& \text{kid\_score}_i \sim N(\mu_i, \sigma^2 = 20^2), \quad i = 1,\ldots, N = 1e6 \\
& \mu_i = 80 + 15 \text{mom\_hs}_i, \quad \text{mom\_hs} \in \{0, 1\}
\end{align*}
$$ {#eq-simulation-anova-model}

This model takes the sample of children from @sec-simple-anova as an *inspiration* but our goal is to focus on the statistical properties of the OLS estimators and not on the data itself. Basically, we will study an artificially created *population* that looks somewhat like the sample of children in the previous example. However, the insights gained in the next sections are more general and are not tied to that specific sample.



In [ ]:
## Population parameters

beta0 <- 80 # Average IQ of children with mom_hs = 0
beta1 <- 15 # Difference in average IQ between children with mom_hs = 1 and mom_hs = 0
sigma <- 20 # Standard deviation of the IQ scores
prop_ones <- 0.8 # Proportion of children with mom_hs = 1

## Population
set.seed(123)

N <- 5e6 # Number of children in the population

pop <- tibble(
  mom_hs = rbinom(N, 1, prop_ones),
  mu = beta0 + beta1 * mom_hs,
  kid_score = rnorm(N, mu, sigma)
) %>%
  select(-mu)



## Selecting the sample

We will select $R = 1000$ samples from the population. Each sample will have $n = 434$ children. We will store the samples in a data frame called `sim`.



In [ ]:
## Fix the random numbers generator so that you can reproduce your results
set.seed(123)

n <- 434 # Sample size
R <- 1000 # Number of samples

sim <- pop %>%
  slice_sample(n = n * R, replace = TRUE) %>%
  mutate(
    # We will add a column to identify the sample number
    sample_id = rep(1:R, each = n)
  )



The resulting data frame `sim` has $434 \times 1000 = 434000$ rows and contains the samples that we will use to study the statistical properties of the OLS estimators.

## OLS estimates in each sample

Now that we have the samples in the data set `sim_samples` we can compute the OLS estimates for $\beta_0$ and $\beta_1$ in each sample.



In [ ]:
sim_coeffs <- sim %>%
  group_by(sample_id) %>%
  ## The tidy function reformats the output of lm so that it can fit in a data frame
  do(tidy(lm(kid_score ~ 1 + mom_hs, data = .))) %>%
  select(sample_id, term, estimate, std.error, statistic)

In [ ]:
## Creates a separate table with the coefficients for mom_hs
sim_slopes <- sim_coeffs %>%
  filter(term == "mom_hs")



The last code chunk may seem a little bit complex, but is simply groups the `sim_samples` data by the sample number and then runs the `lm` function with the data in each sample. You can verify the results in `sim_coeffs` by running the `lm` function manually with the data from the first sample (of course you can choose another sample). The coefficient estimates in `sim_coeffs` are stored in a column called `estimate`. As there are two coefficients in our model, the column `term` tells you whether a row holds the estimate for the intercept $\hat{\beta}_0$ (`term == "(Intercept)"`) or the slope $\hat{\beta}_1$ (`term == "mom_hs"`).

You can use the `filter` function to select only the observations in the first sample



In [ ]:
sample_1 <- sim %>% filter(sample_id == 1)



Now apply `lm` on that sample and compare the coefficient estimates with the first two values in `sim_coeffs`.



In [ ]:
lm(kid_score ~ 1 + mom_hs, data = sample_1)



::: {#exr-sim-ols-estimates}
Select the second sample from the data set `sim` and compute the OLS estimates for the coefficients $\beta_0$ and $\beta_1$.



In [ ]:

# Uncomment the following lines and replace the dots with the correct code

# ... <- sim %>% filter(... == ...)

# lm(..., data = ...)


:::

## Distribution of the OLS estimates

First we plot the distribution of the slope estimates for each sample. In `geom_point` we add a small random value to each estimate so that we can see all the points (this is what `position = "jitter"` does). Otherwise all estimates would lie on the x-axis and we would not be able to see the individual points.



In [ ]:
#| label: fig-sim-distr-slopes
#| fig-cap: "Sampling distribution of $\\hat{\\beta}_1$. Each dot represents the slope estimate in one sample."

# The red line is drawn at the population value of $\beta_1$: 11.77. The  position of the dots on the y-axis does not convey any meaningful information and it only serves to  disentangle  the points so that they don't overplot.

sim_slopes %>%
  ggplot(aes(x = estimate)) +
  geom_point(
    aes(y = 0),
    position = "jitter",
    size = 1 / 2,
    alpha = 0.5
  ) +
  geom_boxplot(alpha = 0.5) +
  ## Draws a density estimate
  geom_density(color = "steelblue") +
  ## Draws a vertical line a 11.77 (the value of beta_1 used in the simulation)
  geom_vline(xintercept = 15, color = "firebrick") + 
  ## Sets the labels of the x and y axis and the title of the plot
  labs(
    x = "Estimate",
    title = "Distribution of the slope estimates in 1000 samples)",
    y = "Density"
  )



The plot reveals three key insights:

1.  The estimates for the slope ($\beta_1$) vary from sample to sample
2.  In most of the samples the estimate was close to the the real value of $\beta_1$ (15)
3.  There is a small number of samples that resulted in extreme values of the estimate.

We can estimate the center (expected value) of this distribution by computing the mean estimate (i.e. the average estimate over all generated samples).



In [ ]:
mean(sim_slopes$estimate)



We see that this value is very close to the real value of 11.77. This is a consequence of a property of the OLS estimator which we call *unbiasedness* [@thm-ols-expected-value]. The average estimate computed above estimates the expected value of the distribution of $\hat{\beta}_1$: $E\hat{\beta}_1$.

The standard deviation of this distribution is called the *standard error* of $\hat{\beta}_1$. It describes the spread of the sampling distribution of the estimates.

::: {.callout-note collapse="true"}
## Is this distribution a coincidence?

The distribution of the OLS estimates that we see in the simulation is a consequence of the properties of the OLS estimator and the sampling process. Remember that we are selecting independent samples (this means that the chance to be selected in one sample does not depend on the chance to be selected in another sample) from the same population.

The OLS estimator is unbiased, meaning that the average over all possible samples is equal to the true value of the population parameter.

We see this in the simulation: the center of the distribution of the estimates is close to the true value of $\beta_1$. We can also show this mathematically.

Remember that the formula for the OLS estimator is

$$
\hat{\beta}_1 = \frac{\sum_{i = 1}^n (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i = 1}^n (x_i - \bar{x})^2}
$$

and remember that the average deviation of a bunch of values from their average is zero.

$$
\sum_{i = 1}^n (x_i - \bar{x}) = 0
$$

This means that we can express the numerator of the OLS estimator as

$$
\sum_{i = 1}^n (x_i - \bar{x})(y_i - \bar{y}) = \sum_{i = 1}^n (x_i - \bar{x})y_i
$$

and the denominator as

$$
\sum_{i = 1}^n (x_i - \bar{x})^2 = \sum_{i = 1}^n (x_i - \bar{x})x_i
$$

Then we can write the OLS estimator as

$$
\hat{\beta}_1 = \frac{\sum_{i = 1}^n (x_i - \bar{x})y_i}{\sum_{i = 1}^n (x_i - \bar{x})x_i}
$$

and more compactly as

$$
\hat{\beta}_1 = \sum_{i = 1}^n w_i y_i
$$

where $w_i = (x_i - \bar{x}) / \sum_{i = 1}^n (x_i - \bar{x})x_i$.

Now we can compute the expected value of the OLS estimator. The expected value of a sum of random variables is the sum of the conditional expected values of the random variables (see @thm-exp-value-props).

$$
E(\hat{\beta}_1 | x) = \sum_{i = 1}^n (E w_i y_i | x)
$$

Because the $w_i$ are fixed (not random) under the condition $x$, we can take them out of the expectation operator.

$$
E(\hat{\beta}_1 | x) = \sum_{i = 1}^n w_i E(y_i | x)
$$

The conditional expectation of $y_i$ is the population regression function $E(y_i | x) = \beta_0 + \beta_1 x_i$. Therefore

$$
E(\hat{\beta}_1 | x) = \sum_{i = 1}^n w_i (\beta_0 + \beta_1 x_i) = \beta_0 \sum_{i = 1}^n w_i + \beta_1 \sum_{i = 1}^n w_i x_i
$$

Now we need to notice that the weights $w_i$ sum to one:

$$
\sum_{i = 1}^n w_i = \sum_{i = 1}^n \frac{x_i - \bar{x}}{\sum_{i = 1}^n (x_i - \bar{x})x_i} = \frac{\sum_{i = 1}^n x_i - n \bar{x}}{\sum_{i = 1}^n (x_i - \bar{x})x_i} = 0
$$

Therefore the expected value of the OLS estimator is

$$
E(\hat{\beta}_1 | x) = \beta_1
$$

We say that the OLS estimator is unbiased. This means that the average of the estimates over all possible samples is equal to the true value of the population parameter (this is basically the result that we saw in the simulation).

Right now we have established that the expected value of the OLS estimator is equal to the true value of the population parameter. There is another property of the OLS estimator (or any other estimater) that is important and that is its variablity (spread) around the expected value over all possible samples.

We can measure the variability of the OLS estimator by computing the standard deviation of the estimates in the simulation. The standard deviation is just the square root of the variance, so we start with the variance.

$$
Var(\hat{\beta}_1 |x) = Var\left(\sum_{i = 1}^n w_i y_i |x \right) = \sum_{i = 1}^n w_i^2 Var(y_i |x) = \sigma^2 \sum_{i = 1}^n w_i^2
$$ {#eq-variance-ols-estimator-slope}

The above expression uses the fact that the $y_i$ are uncorrelated given $x$ and have the same variance $\sigma^2$ (homoscedasticity) and the fact that the weights $w_i$ are fixed (not random) under the condition $x$. It furthermore uses the fact that the variance of a sum of *uncorrelated* random variables is the sum of the variances of these random variables.

$$
Var(X + Y) = Var(X) + Var(Y)
$$

@eq-variance-ols-estimator-slope already provides a formula for the variance of the OLS estimator but it is helpful to simplify it a bit.

The weights $w_i$ are defined as

$$
w_i = \frac{x_i - \bar{x}}{\sum_{i = 1}^n (x_i - \bar{x})x_i}
$$

The square of the weights is

$$
w_i^2 = \left(\frac{x_i - \bar{x}}{\sum_{i = 1}^n (x_i - \bar{x})x_i}\right)^2
$$

and we already know that the denominator is just the sum of squared deviations of $x$.

So we can denote the denominator as

$$
S_{xx} = \sum_{i = 1}^n (x_i - \bar{x})^2 = \sum_{i = 1}^n (x_i - \bar{x})x_i
$$

and rewrite the weights as
:::



In [ ]:
sd(sim_slopes$estimate)



## Hypothesis testing

In @sec-simple-anova we tested the hypothesis that $\beta_1 = 0$ vs $\beta_1 \neq 0$ and talked about a t-statistic, a t-distribution and about critical values derived from that distribution. In the present section our goal is to demystify all these words.

We begin with a simple test of hypotheses about the population value of one of the regression coefficients. Let us start with $\beta_1$ and let us suppose that we want to test the theory that the difference between the average IQ score of the two groups *in the population* equals exactly 11.77. Notice that this is the value of $\beta_1$ that we used in the simulation, so this theory is correct. We also want to test this theory against the alternative that the difference between the average IQ scores is less than 11.77.

### Testing a true null hypothesis {#sec-testing-true-h0}

$$
H_0: \beta_1 \geq 15\\
H_1: \beta_1 < 15
$$ {#eq-hypothesis-true-one-sided}

The mathematical statistics informs us how to summarize the data so that we can make a decision whether to reject the null hypothesis. The t-statistic is the summary that we need for this test. In general it is the difference between the estimate for $\beta_1$ and the value under the null hypothesis divided by the standard error of the estimate.

$$
\text{t-statistic} = \frac{\hat{\beta}_1 - \beta_1^{H_o}}{se(\hat{\beta}_1)}
$$

In our case the value of $\beta_1$ under the null hypothesis is $15$ so the test statistic becomes:

$$
\text{t-statistic} = \frac{\hat{\beta}_1 - 15}{se(\hat{\beta}_1)}.
$$

Notice that this statistic can be large (far away from zero) in two cases:

1.  The null hypothesis is not true. As we have seen in @fig-sim-distr-slopes, the estimates $\hat{\beta}_1$ vary around the true value of $\beta_1$ and this behavior *does not depend on the hypothesis* that we are testing.
2.  Large values of the t-statistic can happen by chance alone, even if the null hypothesis is true. Such large values occur more often (in more samples) when the distribution of $\hat{\beta}_1$ has a high standard deviation.

Let us compute the t-statistic for the hypothesis pair in [-@eq-hypothesis-true-one-sided]. In the data set `slopes` the column `estimate` holds the estimated $\beta_1$, the (estimate) standard error is in the column `std.error`. @fig-sim-t-statistic-h0-true shows the distribution of the t-statistics in our simulated samples.



In [ ]:
# sim_slopes <- sim_slopes %>%
#   mutate(
#     t_statistic = ...
# )

In [ ]:
#| label: fig-sim-t-statistic-h0-true
#| fig-cap: "Distribution of the t-statistic when the null hypothesis is correct."

# sim_slopes %>%
#   ggplot(aes(x = ...)) +
#   geom_point(
#     aes(y = 0),
#     position = "jitter",
#     size = 1 / 2,
#     alpha = 0.5
#   ) +
#   geom_boxplot(alpha = 0.5) +
#   labs(
#     x = "Value of the t-statistic",
#     title = "Distribution of t-statistic under a true null hypothesis beta_1 = 11.77 (2000 samples)",
#     y = ""
#   ) +
#   geom_density(color = "steelblue") +
#   geom_vline(xintercept = 0, color = "red") +
#   geom_vline(xintercept = c(-2), color = "steelblue", lty = 2) +
#   geom_vline(xintercept = c(-3), color = "firebrick", lty = 2) +
#   xlim(c(-4, 8)) +
#   scale_x_continuous(breaks = c(-3, -2, 0))



We should see two things in @fig-sim-t-statistic-h0-true.

1.  First, the t-statistic is different in each sample, even though all samples come from one and the same population (model).
2.  The center of the distribution is close to zero. We should have expected this, because we know that the null hypothesis in [-@eq-hypothesis-true-one-sided] is true.
3.  In some samples the t-statistic has large negative values. In one sample it is even less than -3.

For the hypothesis pair in [-@eq-hypothesis-true-one-sided], large negative values of the t-statistic are consistent with the alternative. Small values close to zero are consistent with the null hypothesis. However, we can see in @fig-sim-t-statistic-h0-true that large negative values of the t-statistic can happen by pure change chance even under a true null hypothesis. Because such extreme values can occur by chance, any decision rule that rejects the null hypothesis for t-statistics lower than some threshold will inevitably produce wrong decision (rejection of the null hypothesis even if it is true).

Let us study the two decision rules depicted as vertical dashed lines in @fig-sim-t-statistic-h0-true. The first rule (blue) rejects the null hypothesis in samples with a t-statistic less than -2. The second decision rule rejects the null hypothesis for t-statistics less than -3.

In the simulation we can simply count the number of samples where we will make a wrong decision to reject the null hypothesis (remember, in this example $H_0$ is true).



In [ ]:
# sim_slopes <- sim_slopes %>%
#   mutate(
#     wrong_2 = ...,
#     wrong_3 = ...
#   )

# ## Share of TRUE values (blue)

## Share of TRUE values (red)



The "less than -2" rule makes a wrong decision in 7 out of 200 samples (3.5 percent). The "less than -3" rule makes a wrong decision in 2 out of 200 samples (one percent).

### Testing a wrong null hypothesis {#sec-testing-wrong-h0}

Now that we've seen the distribution of the t-statistic under a true null hypothesis, let's now examine its distribution under a wrong null hypothesis.

The following hypothesis is wrong in our simulation (because the real $\beta_1 = 11.77$).

$$
H_0: \beta_1 \leq 0\\
H_1: \beta_1 > 0
$$ {#eq-null-wrong}

We use the same statistic for testing this hypothesis. The only difference now is that the alternative is in the other direction. Therefore we would reject for large positive values of the t-statistic.

$$
\text{t-statistic} = \frac{\hat{\beta}_1 - 0}{SE(\hat{\beta}_1)}
$$ {#eq-t-statistic-h0-wrong}

We will compute the t-statistic in [-@eq-t-statistic-h0-wrong] for each sample and will visualize its distribution in @fig-t-statistic-h0-false.



In [ ]:
sim_slopes <- sim_slopes %>%
  mutate(
    t_statistic0 = estimate / std.error
  )

In [ ]:
#| label: fig-t-statistic-h0-false
#| fig-cap: "Distribution of the t-statistic under the (wrong) null hypothesis $\beta_1 = 0$."

# sim_slopes %>%
#   ggplot(aes(x = ...)) +
#   geom_point(
#     aes(y = 0),
#     position = "jitter",
#     size = 1 / 2,
#     alpha = 0.5
#   ) +
#   geom_boxplot(alpha = 0.5) +
#   labs(
#     x = "t-statistic",
#     y = "Density"
#   ) +
#   geom_density(color = "steelblue") +
#   geom_vline(xintercept = 0, color = "red") +
#   geom_vline(xintercept = c(2), color = "steelblue", lty = 2) +
#   geom_vline(xintercept = c(3), color = "firebrick", lty = 2) +
#   xlim(c(-4, 8)) +
#   scale_x_continuous(breaks = c(0, 2, 3))



Notice that the center of the distribution of the t-statistic is no longer close to zero. This happens because the null hypothesis is wrong in this example.

Let us consider two decision rules for rejecting $H_0$. The first one rejects for t-statistics larger than 2, the other one rejects for t-statistics larger than 3.

We can count the number of samples where these two rules lead to wrong decisions. A wrong decision in this case is the non-rejection of the null hypothesis.



In [ ]:
# sim_slopes <- sim_slopes %>%
#   mutate(
#     ## A wrong decision here is to not-reject the null
#     wrong_decision_blue_1 = ...,
#     wrong_decision_red_1 = ...
#   )

## Number of TRUE values

## Share of TRUE values



The rejection rule "greater than 2" rejected the null hypothesis in all samples. The rejection rul "greater than 3" failed to reject the null hypothesis in 8 out of 200 samples (four percent).

## The t-distribution

The mathematical statistics provides us with a theorem that states that under certain conditions (we will talk more about these assumptions) the t-statistic follows a (central) t-distribution with degrees of freedom equal to $n - p$ where $n$ is the number of observations in the sample and $p$ is the number of model coefficients (betas) in the model.

The t-distributions relevant for us have zero mean and the variance determined by the the parameter of the distribution (the degrees of freedom). T-distributions with low degrees of freedom place less probability at the center (zero) and more in the tails of the distribution (extreme values far away from the center) [@fig-t-distribution-df].



In [ ]:
#| label: fig-t-distribution-df
#| fig-cap: "Densities of t-distributions with different degrees of freedom (df)."

dt <- expand_grid(
  ## Creates a sequence of 100 numbers between -3 and 3
  x = seq(-4, 4, length.out = 200),
  df = c(1, 5, 50, 500)
) %>%
  mutate(
    ## Computes the standard normal density at each of the 100 points in x
    t_dens = dt(x, df = df),
    df = factor(df)
  )

ggplot() +
  ## Draws the normal density line
  geom_line(data = dt, aes(x = x, y = t_dens, colour = df)) + 
  labs(
    y = "Density"
  )



Knowing the distribution of the test statistic under the null hypothesis (if it is true) allows us to derive critical values (boundaries for the rejection rules) for the tests. These critical values depend on the type of the alternative in the test.

For a hypothesis pair of the type considered in @sec-testing-true-h0 we reject for negative values ot the t-statistic.

$$
H_0: \beta_1 \geq \beta_1^{H_0}\\
H_1: \beta_1 < \beta_1^{H_0}
$$ {#eq-alternative-one-sided-left}

We want to choose the critical value that meets a predetermined probability of wrong rejection of a true null hypothesis.



In [ ]:
#| label: fig-critical-value-left
#| fig-cap: "Critical value for a one-sided alternative (left)."

dt <- data.frame(
  ## Creates a sequence of 100 numbers between -3 and 3
  x = seq(-4, 4, length.out = 2000)
) %>%
  mutate(
    ## Computes the standard normal density at each of the 100 points in x
    t_dens = dt(x, df = 434 - 2)
  )

ggplot() +
  ## Draws the normal density line
  geom_line(data = dt, aes(x = x, y = t_dens)) +
  ## Draws the shaded area under the curve between
  ## -1 and 1
  geom_ribbon(
    data = filter(dt, x < -1.64),
    aes(x = x, y = t_dens, ymin = 0, ymax = t_dens),
    ## Controls the transparency of the area
    alpha = 0.5
  ) +
  labs(
    x = "t-statistic",
    y = "Density"
  ) +
  annotate(
    "text",
    x = -3.2,
    y = dnorm(0) / 6,
    label = latex2exp::TeX("$P(t < t_{\\alpha}(n - p)) = \\alpha$"),
    parse = TRUE
  ) + 
  geom_vline(xintercept = c(-1.64), lty = 2, colour = "steelblue") +
  # geom_density(data = slopes, aes(x = t_statistic), color = "steelblue4") +
  scale_x_continuous(breaks = c(-1.64, 0), labels = c(latex2exp::TeX("$t_{\\alpha}(n - p)$"), "0"))



If we choose the error probability to be $\alpha$, then the critical value for the one-sided alternative in [-@eq-alternative-one-sided-left] is the $\alpha$ quantile of the t-distribution. We write $t_{\alpha}(n - p)$ to denote this quantile.

In the simulation we had sample sizes $n = 434$ and two coefficients in the model ($\beta_0, \beta_1$), therefore $p = 2$. Let $\alpha = 0.05$. We can compute the quantile using the `qt` function.



In [ ]:
qt(0.05, df = 434 - 2)



The 0.05 quantile of a t-distribution with $432$ degrees of freedom is $t_{0.05}(432) = -1.64$. Therefore the probability to observe samples with a t-statistic less than -1.64 (under a true) null hypothesis is 0.05.

Lets consider another alternative.

$$
H_0: \beta_1 \leq \beta_{H_0}\\
H_1: \beta_1 > \beta_{H_0}
$$

The t-statistic is the same as in the previous test. This time we reject for t-statistics that are large (and positive), because these are the values expected under the alternative.



In [ ]:
#| label: fig-critical-value-right
#| fig-cap: "Critical value for a one-sided alternative (right)."

dt <- data.frame(
  ## Creates a sequence of 100 numbers between -3 and 3
  x = seq(-4, 4, length.out = 2000)
) %>%
  mutate(
    ## Computes the standard normal density at each of the 100 points in x
    t_dens = dt(x, df = 434 - 2)
  )

ggplot() +
  ## Draws the normal density line
  geom_line(data = dt, aes(x = x, y = t_dens)) +
  ## Draws the shaded area under the curve between
  ## -1 and 1
  geom_ribbon(
    data = filter(dt, x > 1.64),
    aes(x = x, y = t_dens, ymin = 0, ymax = t_dens),
    ## Controls the transparency of the area
    alpha = 0.5
  ) +
  labs(
    x = "t-statistic",
    y = "Density"
  ) +
  annotate(
    "text",
    x = 3.2,
    y = dnorm(0) / 6,
    label = latex2exp::TeX("$P(t > t_{1 - \\alpha}(n - p)) = \\alpha$"),
    parse = TRUE
  ) + 
  geom_vline(xintercept = c(1.64), lty = 2, colour = "steelblue") +
  # geom_density(data = slopes, aes(x = t_statistic), color = "steelblue4") +
  scale_x_continuous(breaks = c(0, 1.64), labels = c("0", latex2exp::TeX("$t_{1 - \\alpha}(n - p)$")))



This time the critical value of the $1 - \alpha$ quantile of the t-distribution.

For $df = 432$ and $\alpha = 0.05$ this quantile is $t_{0.95}(432) = 1.64$.



In [ ]:
qt(1 - 0.05, df = 434 - 2)



Due to the symmetry of the t-distribution it is equal to the $0.05$ quantile in absolute value.

Finally, let us consider a two sided alternative.

$$
H_0: \beta_1 = \beta_1^{H_0} \\
H_1: \beta_1 \neq \beta_1^{H_0}
$$

In this test we would reject $H_0$ both for very large positive values and for large (in absolute value) negative values of the t-statistic. Again, we can obtain the critical values from the t-distribution. Because we reject for both positive and negative values, we choose the critical value so that the *total* probability of t-statistics greater than the upper and less than the lower critical value is $\alpha$.



In [ ]:
#| label: fig-critical-value-two-sided
#| fig-cap: "Critical values for a two-sided alternative."

dt <- data.frame(
  ## Creates a sequence of 100 numbers between -3 and 3
  x = seq(-4, 4, length.out = 2000)
) %>%
  mutate(
    ## Computes the standard normal density at each of the 100 points in x
    t_dens = dt(x, df = 434 - 2)
  )

ggplot() +
  ## Draws the normal density line
  geom_line(data = dt, aes(x = x, y = t_dens)) +
  ## Draws the shaded area under the curve between
  ## -1 and 1
  geom_ribbon(
    data = filter(dt, x > 1.64),
    aes(x = x, y = t_dens, ymin = 0, ymax = t_dens),
    ## Controls the transparency of the area
    alpha = 0.5
  ) +
  geom_ribbon(
    data = filter(dt, x < -1.64),
    aes(x = x, y = t_dens, ymin = 0, ymax = t_dens),
    ## Controls the transparency of the area
    alpha = 0.5
  ) +
  labs(
    x = "t-statistic",
    y = "Density"
  ) +
  annotate(
    "text",
    x = -3.2,
    y = dnorm(0) / 6,
    label = latex2exp::TeX("$P(t < t_{\\alpha / 2}(n - p)) = \\alpha / 2$"),
    parse = TRUE
  ) + 
  annotate(
    "text",
    x = 3.2,
    y = dnorm(0) / 6,
    label = latex2exp::TeX("$P(t > t_{1 - \\alpha / 2}(n - p)) = \\alpha / 2$"),
    parse = TRUE
  ) + 
  geom_vline(xintercept = c(-1.64, 1.64), lty = 2, colour = "steelblue") +
  scale_x_continuous(
    breaks = c(-1.64, 0, 1.64), 
    labels = c(
      latex2exp::TeX("$t_{\\alpha / 2}(n - p)$"),
      "0", 
      latex2exp::TeX("$t_{1 - \\alpha / 2}(n - p)$")
    )
  )

In [ ]:
#



For $df = 432$ and $\alpha = 0.05$ the critical values are $t_{0.05 / 2}(432) = -1.96$ and $t_{1 - 0.05 / 2}(432) = 1.96$. Due to the symmetry of the t-distribution these critical values are equal in absolute value.

### The p-value

Another way to decide whether to reject a null hypothesis or not is to compute the p-value of the test. The p-value is conditional the probability (under the null hypothesis) to see samples with a t-statistic that is even more extreme than the one observed in a sample. How the p-value is calculated depends on the alternative in the test.

Let us take the example with the children from @sec-simple-anova. For convenience we print here again the output from `summary`.



In [ ]:
summary(lm(kid_score ~ 1 + mom_hs, data = kids))



Now we consider three null-hypothesis and alternative pairs.

$$
H_0: \beta_1 \leq 0 \\
H_0: \beta_1 > 0
$$ The sample value of the t-statistic is

$$
\frac{11.77 - 0}{2.3} \approx 5.1
$$

For this alternative extreme (consistent with the alternative) values of the t-statistic are large positive values. Therefore the p-value of this test is

$$
P(t > \text{t-statistic}^{obs} | H_0)
$$

The notation $\text{t-statistic}^{obs}$ refers to the value of the statistic in the sample (5.1). If $H_0$ is true, i.e. $\beta_1 = 0$ we know that the t-statistic follows a t-distribution with $df = 432$. We can compute the probability of samples with t-statistics less than 5.1 using the `pt` function.



In [ ]:
## Calculate the p-value for the one-sided alternative
#



This probability is equal to about 2.5 out of 10 million. This means that samples with a t-statistic greater than the one observed (5.1) are extremely rare. Either we have been extremely lucky to select a very rare sample or the null hypothesis is wrong.

Let us now consider the other one-sided alternative

$$
H_0: \beta_1 \geq 0 \\
H_1: \beta_1 < 0
$$

For this alternative extreme values of the t-statistic are large (in absolute value) negative values. Therefore the p-value of the test is

$$
P(t < \text{t-statistic}^{obs} | H_0)
$$ We can compute this probability using `pt`.





This p-value is close to one, implying that under the null hypothesis almost all samples would have a t-statistic less than 5.1.

Finally, let us consider the two-sided alternative

$$
H_0: \beta_1 = 0 \\
H_1: \beta_1 \neq 0
$$

Now both large positive and large (in absolute value) negative values of the t-statistic are extreme (giving evidence against the null hypothesis). The p-value in this case is the sum of two probabilities. For $t = 5.1$ more extreme values against the null hypothesis are the ones greater than 5.1 and the ones less than -5.1.

$$
\text{p-value} = P(t < -5.1 | H_0) + P(t > 5.1 | H_0)
$$

We can compute it with a the `pt` function. Notice that this is the same value (up to rounding errors) as the one shown in the $Pr(>|t|)$ column in the output of `summary`.





How would the p-value look like if the t-statistic was negative, e.g. -5.1? Again, extreme values are the ones less that -5.1 and greater than the absolute value, i.e. 5.1. We can write this more succinctly (and this explains the notation for the column in the `summary` output):

$$
\text{p-value} = P(|t| > |\text{t-statistic}^{obs}| |H_0)
$$

Finally, let us see the distribution of p-values in the simulation study. Here we will compute it for the true null hypothesis $H_0: \beta_1 = 11.77, H_1: \beta_1 \neq 11.77$. Figure @fig-p-value-h0-true shows the estimated density of the p-values (in each sample). You can see that the density is roughly constant over the interval \[0, 1\]. Indeed, it can be shown that the p-value if uniformly distributed over this interval if the null hypothesis is true.



In [ ]:
# sim_slopes <- sim_slopes %>%
#   mutate(
#     p_value_h0_true = ...
#   )

In [ ]:
#| label: fig-p-value-h0-true
#| fig-cap: "Distribution of p-values under the (true) null hypothesis"

# sim_slopes %>%
#   ggplot(aes(x = ...)) + 
#   geom_density() + 
#   labs(
#     x = "p-value",
#     y = "Density"
#   )



When using the p-value to make a rejection decision we compare it to a predetermined probability of a wrong rejection of a true null hypothesis. This is the same approach that we followed when we derived the critical values of the tests. A widely used convention is to reject a null hypothesis if the p-value is less than 0.05.

Let us apply this convention in the simulation and see in how many samples we make a wrong decision.



In [ ]:
# sim_slopes <- sim_slopes %>%
#   mutate(
#     reject_h0_based_on_p_value = ...
#   )

# table(sim_slopes$reject_h0_based_on_p_value)



## Confidence intervals {#sec-sim-ci}

From the distribution of the t-statistic we can see that

$$
\text{t-statistic} = \frac{\hat{\beta}_1 - \beta_1}{se(\hat{\beta}_1)} \sim t(n - p)
$$

Therefore the probability to observed a value of the t-statistic in the interval \[$t_{\alpha / 2}(n - p)$, $t_{1 - \alpha / 2}$\] is $1 - \alpha$.

$$
P\left(t_{\alpha / 2}(n - p) < \frac{\hat{\beta}_1 - \beta_1}{se(\hat{\beta}_1)} < t_{1 - \alpha / 2}(n - p)\right) = \\
P\left(\hat{\beta_1} + se(\hat{\beta}_1) t_{\alpha/2}(n - p) < \beta_1 < \hat{\beta_1} + se(\hat{\beta}_1) t_{1 - \alpha/2}(n - p) \right) = 1 - \alpha
$$

Compute the upper and the lower bounds of the confidence intervals for each sample in the simulation with a coverage probability of $1 - \alpha = 0.9$. In how many samples did the confidence interval contain the real coefficient $\beta_1$?



In [ ]:
# sim_slopes <- sim_slopes %>%
#   mutate(
#     CI_lower = ...,
#     CI_upper = ...,
#     ## Construct a new variables that is TRUE/FALSE if the
#     ## true value of beta_1 (11.77) is inside the confidence interval
#     beta1_in_CI = ...
#   )

In [ ]:
# sum(sim_slopes$beta1_in_CI)
# mean(sim_slopes$beta1_in_CI)



The CI contained the true value of $\beta_1$ in 178 out of 200 samples (89 percent). This is quite close to the coverage probability $1 - \alpha = 0.9$ The CI that we constructed above are visualized in @fig-sim-ci. For the sake of clarity the plot is limited to the first 50 samples. Confidence intervals that do not contain the true value of $\beta_1$ are show in red. Notice that the boundaries differ from sample to sample (as to the coefficient estimates).



In [ ]:
#| label: fig-sim-ci
#| fig-cap: "Confidence intervals for the first 50 samples. The vertical line is drawn at 11.77 (the real value of $\\beta_1$)."

# sim_slopes %>%
#   ungroup() %>%
#   slice_head(n = 50) %>%
#   ggplot(
#     aes(
#       x = estimate, 
#       y = factor(R),
#       xmin = CI_lower,
#       xmax = CI_upper,
#       color = beta1_in_CI
#     )
#   ) + 
#   geom_point() + 
#   geom_errorbarh() + 
#   labs(
#     x = "",
#     y = "Sample",
#     color = "Beta1 in CI"
#   ) + 
#   theme(
#     axis.text.y = element_blank(),
#     axis.ticks.y = element_blank(),
#   ) + 
#   geom_vline(
#     xintercept = 15
#   )

In [ ]:
# dt <- tibble(
#   x = 1:10,
#   y = rnorm(10, mean = 2 + 0.3 * x, sd = 0.5)
# )

# fit <- lm(y ~ 0 + x, data = dt)
# sum(residuals(fit))